# Tables

All tables for the paper. Each is displayed here and written to `../tables/` as `.csv` and a booktabs `.tex` fragment.

Most results are read from the CSVs written by `02_training_and_testing_the_ann.ipynb` and `03_ann_vs_linear_models.ipynb` (in `tables/`, next to the tables built from them). The free-slope circular (Möbius) baselines, and the leave-one-wind-speed-out baselines, are not among those results, so they are fit here with the same trajectory-grouped split (seed 42, 33% test) and the same evaluation protocol as the OLS baselines.

1. Linear / circular baselines and the ANN, by regime
2. Per-trajectory error distribution, all models
3. ANN error by regime
4. Leave-one-wind-speed-out (ANN and baselines)
5. Robustness across training seeds
6. Architecture selection (grouped CV)
7. Ensemble-agreement calibration
8. ANN error vs. trajectory covariates

In [1]:
import os, sys
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import spearmanr
from sklearn.model_selection import GroupShuffleSplit
sys.path.insert(0, os.path.abspath('../utils'))
from utils import circular_distance
from IPython.display import display

D = '../trainingdata/04_trainingdata/'
OUT = '../tables/'
os.makedirs(OUT, exist_ok=True)
pd.set_option('display.width', 200); pd.set_option('display.max_columns', None)
REGIMES = ['upwind', 'crosswind', 'downwind']
THRESH = 15.0

def classify_direction(a):
    deg = np.degrees(np.arctan2(np.sin(a), np.cos(a)))
    return np.where(np.abs(deg) <= 45, 'downwind', np.where(np.abs(np.abs(deg) - 180) <= 45, 'upwind', 'crosswind'))

def bootstrap_ci(v, n_boot=5000, seed=0):
    v = np.asarray(v); rng = np.random.default_rng(seed)
    med = np.median(rng.choice(v, size=(n_boot, len(v)), replace=True), axis=1)
    lo, hi = np.percentile(med, [2.5, 97.5])
    return float(np.median(v)), float(lo), float(hi)

def ci_str(m, lo, hi): return f'{m:.2f} [{lo:.2f}, {hi:.2f}]'

def esc(s): return str(s).replace('_', r'\_').replace('%', r'\%').replace('&', r'\&')

def save(df, name, caption, label, align=None):
    df.to_csv(f'{OUT}{name}.csv', index=False)
    align = align or 'l' + 'r' * (df.shape[1] - 1)
    lines = [r'\begin{table}[h!]', r'    \centering', f'    \\caption{{{caption}}}', f'    \\label{{tab:{label}}}',
             f'    \\begin{{tabular}}{{@{{}}{align}@{{}}}}', r'        \toprule',
             '        ' + ' & '.join(esc(c) for c in df.columns) + r' \\', r'        \midrule']
    for _, r in df.iterrows():
        lines.append('        ' + ' & '.join(esc(x) for x in r.values) + r' \\')
    lines += [r'        \bottomrule', r'    \end{tabular}', r'\end{table}']
    open(f'{OUT}{name}.tex', 'w').write('\n'.join(lines) + '\n')
    display(df)

/Users/florisvb/src/drosophila_body_orientation_predictor/.venv/lib/python3.12/site-packages/pynumdiff/__init__.py:9: UserWarning: tvrdiff, robustdiff, and lineardiff not available due to lack of convex solver. To use those, install CVXPY.
  warn("tvrdiff, robustdiff, and lineardiff not available due to lack of convex solver. To use those, install CVXPY.")


## Setup: split, and linear / circular / Möbius baselines

Baselines: *linear* OLS regresses the heading unit vector on the Cartesian components of each kinematic vector; *circular* OLS regresses it on the unit direction only (degree-1 trigonometric polynomial). The additional **Möbius** model is the free-slope circular-circular regression from `03_ann_vs_linear_models.ipynb`: $\hat\theta = \mu + 2\arctan(\beta\tan\frac{\theta_x-\nu}{2})$, fit by minimising $\sum(1-\cos e)$. It takes a single input angle, so it is fit once per kinematic vector.

In [2]:
data = pd.read_csv(D + 'all_wind_heading_and_trajectories_augmented_corrected_filtered_headx_heady.csv')
ids = data['trajec_objid'].unique()
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.33, random_state=42).split(np.zeros((len(ids), 1)), groups=ids))
train_df = data[data['trajec_objid'].isin(set(ids[tr]))].reset_index(drop=True)
test_df  = data[data['trajec_objid'].isin(set(ids[te]))].reset_index(drop=True)
print(len(train_df), 'train frames,', len(test_df), 'test frames')

QUANT = {'thrust': ('thrust', 'thrust_angle'), 'air': ('airspeed', 'airspeed_angle'), 'ground': ('groundspeed', 'groundspeed_angle')}

def design(d, combo, kind):
    cols = [np.ones(len(d))]
    for q in combo:
        m, a = QUANT[q]
        if kind == 'linear': cols += [d[m].values * np.cos(d[a].values), d[m].values * np.sin(d[a].values)]
        else:                cols += [np.cos(d[a].values), np.sin(d[a].values)]
    return np.column_stack(cols)

def ols_angle(train, test, combo, kind):
    Xtr, Xte = design(train, combo, kind), design(test, combo, kind)
    coef = np.linalg.lstsq(Xtr, train[['heading_angle_x', 'heading_angle_y']].values, rcond=None)[0]
    p = Xte @ coef
    return np.arctan2(p[:, 1], p[:, 0])

def circ_mean(a): return np.arctan2(np.sin(a).mean(), np.cos(a).mean())
def mobius(x, mu, nu, lb): return mu + 2 * np.arctan(np.exp(lb) * np.tan((x - nu) / 2))

def fit_mobius(x, y):
    best = None
    for nu0 in np.linspace(0, 2 * np.pi, 8, endpoint=False):
        for lb0 in (-1.0, 0.0, 1.0, 2.0):
            r = minimize(lambda p: np.mean(1 - np.cos(y - mobius(x, *p))), [circ_mean(y - x), nu0, lb0],
                         method='Nelder-Mead', options=dict(xatol=1e-6, fatol=1e-10, maxiter=4000))
            if best is None or r.fun < best.fun: best = r
    return best.x

def mobius_angle(train, test, q):
    a = QUANT[q][1]
    return mobius(test[a].values, *fit_mobius(train[a].values, train['heading_angle'].values))

def frame_errors(test, pred_angle):
    err = np.degrees(circular_distance(test['heading_angle'].values, pred_angle))
    return pd.DataFrame({'trajec_objid': test['trajec_objid'].values, 'err': err,
                         'regime': classify_direction(test['heading_angle'].values)})

def summarize(fdf):
    """per-trajectory medians (pooled and per regime), as in the paper"""
    pooled = fdf.groupby('trajec_objid')['err'].median()
    by_reg = fdf.groupby(['regime', 'trajec_objid'])['err'].median().reset_index()
    out = {'overall': pooled.median()}
    for r in REGIMES: out[r] = by_reg.loc[by_reg['regime'] == r, 'err'].median()
    return out, pooled, by_reg

# Möbius baselines on the paper's split
mobius_rows, mobius_traj = [], {}
for q in QUANT:
    s, pooled, _ = summarize(frame_errors(test_df, mobius_angle(train_df, test_df, q)))
    name = f'mobius: {q}'
    mobius_rows.append({'model': name, **s}); mobius_traj[name] = pooled
print(pd.DataFrame(mobius_rows))

# sanity check: reproduce one saved OLS baseline
ref = pd.read_csv(OUT + 'ols_baselines_by_regime.csv').set_index('model').loc['linear: thrust+air', 'overall']
chk, _, _ = summarize(frame_errors(test_df, ols_angle(train_df, test_df, ('thrust', 'air'), 'linear')))
print(f"check linear: thrust+air overall: saved {ref:.3f}, recomputed {chk['overall']:.3f}")

75141 train frames, 36942 test frames


            model    overall     upwind  crosswind   downwind
0  mobius: thrust  15.216739  10.870873  38.967758  38.976396
1     mobius: air   8.321766   7.050029  26.318810  21.129393
2  mobius: ground  15.910523   8.469371  43.217996  39.076628
check linear: thrust+air overall: saved 9.633, recomputed 9.633


## Table 1 — Linear, circular and Möbius baselines vs. the ANN, by flight regime

Median per-trajectory angular error (deg) on the held-out test set. Regimes are classified from the true heading relative to the wind axis (downwind $|\theta|\le45^\circ$, upwind within $45^\circ$ of $180^\circ$, otherwise crosswind; the wind blows toward +x).

In [3]:
ann_frames = pd.read_csv(OUT + 'ann_by_regime.csv')
ann_s, ann_pooled, ann_by_reg = summarize(ann_frames)
base = pd.read_csv(OUT + 'ols_baselines_by_regime.csv')
t1 = pd.concat([base, pd.DataFrame(mobius_rows), pd.DataFrame([{'model': 'ANN', **ann_s}])], ignore_index=True)
t1['family'] = t1['model'].str.split(':').str[0].map({'linear': 'Linear OLS', 'circular': 'Circular OLS',
                                                     'mobius': 'Möbius circular regression', 'ANN': 'ANN'})
t1['inputs'] = t1['model'].str.split(': ').str[-1].where(t1['model'] != 'ANN', 'groundspeed + airspeed')
t1 = t1.sort_values('overall', ascending=False).reset_index(drop=True)
table1 = t1[['family', 'inputs', 'overall', 'upwind', 'crosswind', 'downwind']].copy()
table1.columns = ['Model', 'Inputs', 'Overall', 'Upwind', 'Crosswind', 'Downwind']
best_idx = {c: t1[t1['family'] != 'ANN'][c.lower()].idxmin() for c in ['Overall', 'Upwind', 'Crosswind', 'Downwind']}
print('best non-ANN baseline per column:', {c: (t1.loc[i, 'model'], round(t1.loc[i, c.lower()], 2)) for c, i in best_idx.items()})
for c in ['Overall', 'Upwind', 'Crosswind', 'Downwind']: table1[c] = table1[c].map('{:.2f}'.format)
save(table1, 'table1_baselines_by_regime',
     'Median per-trajectory angular error (deg) of every linear, circular and M\\"obius baseline and of the ANN on the held-out test trajectories, overall and by flight regime. Sorted by overall error, worst first.',
     'baseline_regime', align='llrrrr')

best non-ANN baseline per column: {'Overall': ('mobius: air', 8.32), 'Upwind': ('linear: air+ground', 5.51), 'Crosswind': ('mobius: air', 26.32), 'Downwind': ('mobius: air', 21.13)}


,Model,Inputs,Overall,Upwind,Crosswind,Downwind
0,Circular OLS,ground,16.76,7.85,36.32,96.37
1,Möbius circular regression,ground,15.91,8.47,43.22,39.08
2,Möbius circular regression,thrust,15.22,10.87,38.97,38.98
3,Linear OLS,thrust,14.98,9.99,45.69,62.29
4,Circular OLS,thrust,13.88,9.42,48.45,41.52
5,Linear OLS,ground,11.01,5.52,33.86,48.83
6,Circular OLS,thrust+ground,10.71,6.84,34.05,29.59
7,Linear OLS,air+ground,10.24,5.51,33.01,45.45
8,Linear OLS,air,10.04,5.99,33.90,47.44
9,Circular OLS,air,9.79,6.15,42.62,30.97


## Table 2 — Per-trajectory error distribution, all models

Median with 95% bootstrap CI (resampling trajectories), mean, interquartile range, 90th percentile and fraction of trajectories with error above 15°.

In [4]:
def dist_row(name, v):
    v = np.asarray(v); m, lo, hi = bootstrap_ci(v)
    return {'Model': name, 'n': len(v), 'Median [95% CI]': ci_str(m, lo, hi), 'Mean': f'{v.mean():.2f}',
            'IQR': f'{np.percentile(v, 25):.2f}–{np.percentile(v, 75):.2f}', 'Q90': f'{np.percentile(v, 90):.2f}',
            f'% > {THRESH:g}°': f'{100 * (v > THRESH).mean():.1f}'}

per_traj_all = pd.read_csv(OUT + 'trajectory_level_errors_all_models.csv')
rows = [dist_row(m, g['err']) for m, g in per_traj_all.groupby('model')]
rows += [dist_row(m, v.values) for m, v in mobius_traj.items()]
table2 = pd.DataFrame(rows)
table2['_k'] = table2['Median [95% CI]'].str.split(' ').str[0].astype(float)
table2 = table2.sort_values('_k', ascending=False).drop(columns='_k').reset_index(drop=True)
save(table2, 'table2_error_distribution',
     'Distribution of per-trajectory median angular error (deg) on the held-out test set for the ANN and all baselines.',
     'error_distribution', align='lrrrrrr')

,Model,n,Median [95% CI],Mean,IQR,Q90,% > 15°
0,circular: ground,1096,"16.76 [14.52, 19.51]",32.08,4.33–47.30,91.41,52.1
1,mobius: ground,1096,"15.91 [13.53, 18.09]",28.93,5.36–39.31,62.88,51.0
2,mobius: thrust,1096,"15.22 [13.57, 16.70]",22.26,7.71–29.80,50.28,50.2
3,linear: thrust,1096,"14.98 [13.64, 16.59]",24.10,6.98–35.85,56.95,49.8
4,circular: thrust,1096,"13.88 [12.68, 15.45]",22.47,6.54–33.96,53.30,47.6
5,linear: ground,1096,"11.01 [9.56, 12.70]",22.66,4.10–31.05,53.62,43.1
6,circular: thrust+ground,1096,"10.71 [9.84, 12.05]",19.06,4.56–27.46,46.02,41.4
7,linear: air+ground,1096,"10.24 [9.03, 11.68]",20.87,4.05–28.24,49.77,41.1
8,linear: air,1096,"10.04 [8.92, 11.40]",20.66,4.35–29.43,50.27,41.1
9,circular: air,1096,"9.79 [8.57, 10.79]",19.80,4.41–25.86,49.79,40.3


## Table 3 — ANN error by flight regime

Per-trajectory median error within each regime (a trajectory contributes to every regime it visits), and the best baseline in that regime for reference.

In [5]:
def regime_row(label, v):
    v = np.asarray(v); m, lo, hi = bootstrap_ci(v)
    return {'Regime': label, 'n trajectories': len(v), 'Median [95% CI]': ci_str(m, lo, hi), 'Mean': f'{v.mean():.2f}',
            'IQR': f'{np.percentile(v, 25):.2f}–{np.percentile(v, 75):.2f}', f'% > {THRESH:g}°': f'{100 * (v > THRESH).mean():.1f}'}

rows = [regime_row('Overall', ann_pooled.values)]
rows += [regime_row(r.capitalize(), ann_by_reg.loc[ann_by_reg['regime'] == r, 'err']) for r in REGIMES]
table3 = pd.DataFrame(rows)
nb_base = t1[t1['family'] != 'ANN']
table3['Best baseline (median)'] = [f"{nb_base[c].min():.2f}  ({nb_base.loc[nb_base[c].idxmin(), 'model']})"
                                    for c in ['overall'] + REGIMES]
save(table3, 'table3_ann_by_regime',
     'ANN per-trajectory angular error (deg) on the held-out test set, overall and by flight regime, with the best baseline median in each regime for comparison.',
     'ann_regime', align='lrrrrrl')

,Regime,n trajectories,Median [95% CI],Mean,IQR,% > 15°,Best baseline (median)
0,Overall,1096,"4.96 [4.65, 5.44]",14.27,2.65–11.53,20.3,8.32 (mobius: air)
1,Upwind,903,"5.08 [4.80, 5.70]",14.62,2.67–11.23,19.7,5.51 (linear: air+ground)
2,Crosswind,614,"20.16 [16.79, 23.86]",34.59,6.50–55.02,55.9,26.32 (mobius: air)
3,Downwind,242,"5.64 [4.31, 7.27]",35.19,2.46–48.13,33.9,21.13 (mobius: air)


## Table 4 — Leave-one-wind-speed-out

ANN: trained on two wind speeds and tested on the third (read from the saved results). The three baselines are refit here under the same leave-one-out design: the best pooled linear and circular OLS baselines and the airspeed Möbius regression. Baseline errors are medians over all frames of the held-out trajectories; ANN errors use the trajectories after the minimum-length filter, so counts can differ slightly.

In [6]:
lowo = pd.read_csv(OUT + 'leave_one_wind_out_results.csv')
# best linear and best circular OLS by pooled error, chosen as in 03_ann_vs_linear_models.ipynb
best_ols = {kind: base.loc[base.loc[base['model'].str.startswith(kind + ':'), 'overall'].idxmin(), 'model'].split(': ')[1]
            for kind in ('linear', 'circular')}
BASELINES = [(f'Linear ({best_ols["linear"]})', tuple(best_ols['linear'].split('+')), 'linear'),
             (f'Circular ({best_ols["circular"]})', tuple(best_ols['circular'].split('+')), 'circular'),
             ('Möbius (air)', ('air',), 'mobius')]
print('baselines:', [label for label, _, _ in BASELINES])
rows = []
for ws in sorted(data['windspeed'].unique()):
    trn, tst = data[data['windspeed'] != ws], data[data['windspeed'] == ws].reset_index(drop=True)
    res = {}
    for label, combo, kind in BASELINES:
        ang = mobius_angle(trn, tst, combo[0]) if kind == 'mobius' else ols_angle(trn, tst, combo, kind)
        s, _, _ = summarize(frame_errors(tst, ang)); res[label] = s
    for reg in ['overall'] + REGIMES:
        a = lowo[(np.isclose(lowo['held_out_windspeed'], ws)) & (lowo['regime'] == reg)].iloc[0]
        rows.append({'Held-out wind speed (m/s)': ws, 'Regime': reg.capitalize(), 'n (ANN)': int(a['n_trajectories']),
                     'ANN [95% CI]': ci_str(a['median'], a['ci95_lo'], a['ci95_hi']),
                     **{label: f'{res[label][reg]:.2f}' for label, _, _ in BASELINES}})
table4 = pd.DataFrame(rows)
save(table4, 'table4_leave_one_wind_out',
     'Leave-one-wind-speed-out median per-trajectory angular error (deg). The ANN and the three baselines are trained on two wind speeds and tested on the held-out third.',
     'lowo', align='lllrrrr')

baselines: ['Linear (thrust+air+ground)', 'Circular (air+ground)', 'Möbius (air)']


,Held-out wind speed (m/s),Regime,n (ANN),ANN [95% CI],Linear (thrust+air+ground),Circular (air+ground),Möbius (air)
0,0.3,Overall,1266,"6.01 [5.57, 6.41]",17.81,10.90,8.46
1,0.3,Upwind,930,"6.04 [5.59, 6.66]",6.20,6.75,8.01
2,0.3,Crosswind,783,"16.51 [14.27, 19.05]",37.49,33.80,16.70
3,0.3,Downwind,377,"4.99 [4.13, 5.85]",66.85,10.71,14.97
4,0.4,Overall,1060,"4.73 [4.40, 5.20]",9.58,10.18,8.42
5,0.4,Upwind,853,"4.76 [4.32, 5.36]",5.22,6.00,6.45
6,0.4,Crosswind,604,"15.52 [12.86, 18.70]",29.83,32.94,21.53
7,0.4,Downwind,245,"5.27 [4.24, 8.48]",46.34,20.59,22.18
8,0.6,Overall,995,"6.47 [5.88, 7.05]",7.64,9.62,6.95
9,0.6,Upwind,924,"5.36 [5.09, 5.66]",6.18,6.64,5.51


## Table 5 — Robustness across training seeds

In [7]:
seeds = pd.read_csv(OUT + 'ann_multiseed_results.csv')
t5 = pd.DataFrame({'Seed': seeds['seed'], 'Median error (deg)': seeds['median_error'].map('{:.2f}'.format),
                   'Mean error (deg)': seeds['mean_error'].map('{:.2f}'.format)})
summary = pd.DataFrame([{'Seed': 'mean ± sd', 'Median error (deg)': f"{seeds['median_error'].mean():.2f} ± {seeds['median_error'].std():.2f}",
                         'Mean error (deg)': f"{seeds['mean_error'].mean():.2f} ± {seeds['mean_error'].std():.2f}"},
                        {'Seed': 'range', 'Median error (deg)': f"{seeds['median_error'].min():.2f}–{seeds['median_error'].max():.2f}",
                         'Mean error (deg)': f"{seeds['mean_error'].min():.2f}–{seeds['mean_error'].max():.2f}"}])
table5 = pd.concat([t5, summary], ignore_index=True)
save(table5, 'table5_multiseed', 'ANN performance across five independently seeded training runs on the held-out test set.',
     'multiseed', align='lrr')

,Seed,Median error (deg),Mean error (deg)
0,42,4.96,14.27
1,7,5.04,14.14
2,13,5.04,14.03
3,99,5.02,14.27
4,123,4.92,14.22
5,mean ± sd,5.00 ± 0.05,14.19 ± 0.10
6,range,4.92–5.04,14.03–14.27


## Table 6 — Architecture selection (3-fold trajectory-grouped CV)

Out-of-fold median per-trajectory error with 95% bootstrap CI and IQR for each candidate architecture, trained with the geodesic loss and trajectory-length sample weights.

In [8]:
def cv_rows(df, label_fn):
    return [{'Configuration': label_fn(r), 'n': int(r['n_trajectories']),
             'CV median [95% CI]': ci_str(r['cv_median_error'], r['cv_ci95_lo'], r['cv_ci95_hi']),
             'IQR': f"{r['cv_q25']:.2f}–{r['cv_q75']:.2f}"} for _, r in df.iterrows()]

arch = lambda r: f"{int(r['layers'])} × {int(r['neurons'])}"
if os.path.exists(OUT + 'architecture_cv_results.csv'):
    table6 = pd.DataFrame(cv_rows(pd.read_csv(OUT + 'architecture_cv_results.csv'), arch))
    save(table6, 'table6_architecture_selection',
         'Grouped cross-validation results used to select the ANN architecture (geodesic loss, trajectory-length sample weights).',
         'selection', align='lrrr')
else:
    print('Skipping Table 6: architecture_cv_results.csv not found (run 02_training_and_testing_the_ann.ipynb with RETRAIN = True).')

,Configuration,n,CV median [95% CI],IQR
0,3 × 30,2225,"5.13 [4.84, 5.47]",2.72–12.45
1,3 × 20,2225,"5.26 [4.96, 5.58]",2.86–12.69
2,3 × 15,2225,"5.27 [5.00, 5.56]",2.82–12.83
3,2 × 20,2225,"5.27 [5.02, 5.58]",2.91–12.74
4,2 × 15,2225,"5.34 [5.09, 5.66]",2.92–12.69
5,3 × 10,2225,"5.36 [5.13, 5.65]",2.90–13.31
6,1 × 30,2225,"5.47 [5.27, 5.79]",3.00–13.89
7,1 × 20,2225,"5.52 [5.29, 5.78]",3.08–13.53
8,2 × 10,2225,"5.58 [5.32, 5.94]",2.91–13.60
9,2 × 30,2225,"5.58 [5.33, 5.82]",3.18–12.88


## Table 7 — Ensemble-agreement calibration

In [9]:
ens = pd.read_csv(OUT + 'ann_ensemble_agreement.csv')
ens['quartile'] = pd.qcut(ens['R'], 4, labels=['Q1 (lowest agreement)', 'Q2', 'Q3', 'Q4 (highest agreement)'])
table7 = ens.groupby('quartile', observed=True).agg(n=('err', 'size'), median_R=('R', 'median'),
                                                     median_err=('err', 'median'), mean_err=('err', 'mean')).reset_index()
rho = spearmanr(ens['R'], ens['err']).statistic
table7.columns = ['Agreement quartile', 'n', 'Median R', 'Median error (deg)', 'Mean error (deg)']
table7['Median R'] = table7['Median R'].map('{:.5f}'.format)
for c in ['Median error (deg)', 'Mean error (deg)']: table7[c] = table7[c].map('{:.2f}'.format)
print(f'Spearman rho (R vs error) = {rho:.3f}')
save(table7, 'table7_ensemble_calibration',
     f'Per-trajectory error by quartile of five-model ensemble mean resultant length $R$ (Spearman $\\rho={rho:.2f}$).',
     'calibration', align='lrrrr')

Spearman rho (R vs error) = -0.602


,Agreement quartile,n,Median R,Median error (deg),Mean error (deg)
0,Q1 (lowest agreement),274,0.99869,14.62,31.25
1,Q2,274,0.99966,6.05,12.29
2,Q3,274,0.99981,4.20,9.56
3,Q4 (highest agreement),274,0.99990,2.40,3.99


## Table 8 — ANN error vs. trajectory covariates

Spearman rank correlation between per-trajectory ANN error and trajectory-level kinematic summaries on the held-out set, and median error by wind speed.

In [10]:
cov = pd.read_csv(OUT + 'ann_error_with_covariates.csv')
labels = {'mean_slip_angle': 'Mean |slip angle| (deg)', 'max_turning_rate': 'Max turning rate (deg/frame)',
          'mean_thrust': 'Mean thrust magnitude', 'mean_groundspeed': 'Mean groundspeed (m/s)',
          'mean_airspeed': 'Mean airspeed (m/s)', 'windspeed': 'Wind speed (m/s)'}
rows = []
for c, lab in labels.items():
    r = spearmanr(cov[c], cov['err'])
    rows.append({'Covariate': lab, 'Spearman ρ': f'{r.statistic:.3f}', 'p': f'{r.pvalue:.1e}', 'n': len(cov)})
table8 = pd.DataFrame(rows).sort_values('Spearman ρ', key=lambda s: s.astype(float).abs(), ascending=False).reset_index(drop=True)
save(table8, 'table8_covariate_correlations',
     'Spearman correlation between per-trajectory ANN error and trajectory covariates on the held-out test set.',
     'covariates', align='lrrr')
ws_tab = cov.groupby('windspeed')['err'].agg(n='size', median='median', mean='mean').round(2).reset_index()
display(ws_tab)

,Covariate,Spearman ρ,p,n
0,Mean |slip angle| (deg),0.703,8.2e-164,1096
1,Max turning rate (deg/frame),0.587,2.7e-102,1096
2,Mean groundspeed (m/s),-0.545,1.1e-85,1096
3,Mean airspeed (m/s),-0.470,2.4e-61,1096
4,Mean thrust magnitude,-0.430,2.0e-50,1096
5,Wind speed (m/s),-0.004,8.9e-01,1096


,windspeed,n,median,mean
0,0.3,396,4.97,14.09
1,0.4,368,4.92,14.19
2,0.6,332,5.15,14.57
